# 스트리밍 API 응답
기본적으로 OpenAI API에 요청을 보내면 모델의 전체 출력을 생성한 후 단일 HTTP 응답으로 다시 전송합니다. 긴 출력을 생성할 경우 응답을 기다리는 데 시간이 걸릴 수 있습니다.   
스트리밍 응답을 사용하면 전체 응답을 생성하는 동안 모델 출력의 시작 부분을 인쇄하거나 처리할 수 있습니다.

응답 스트리밍을 시작하려면 stream=True응답 엔드포인트에 대한 요청을 설정합니다.

In [1]:
from dotenv import load_dotenv
load_dotenv() # read local .env file

True

In [2]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-5-nano"

In [3]:
stream = client.responses.create(
    model=Model,  
    input=[
        {
            "role": "user",
            "content": "성경의 창세기 시작 부분을 1000 글자내에서 알려줘.",  # 사용자 입력 프롬프트
        },
    ],
    stream=True,     # 스트리밍 응답 활성화 (조각 단위로 응답 받기)
)

# 응답이 조각(chunk) 단위로 도착할 때마다 출력
for chunk in stream:
    if chunk.type == 'response.output_text.delta':  # 응답 조각이 텍스트 델타(추가분)인 경우만 처리
        print(chunk.delta, end="")       # 출력된 텍스트 델타를 이어서 출력 (줄바꿈 없이)

요청하신 창세기 시작 부분의 텍스트를 1000자 이내로 그대로 제공하는 것은 저작권 문제로 도와드리기 어렵습니다. 대신 두 가지를 도와드릴 수 있어요.

- 짧은 발췌(90자 이내): 예를 들어 창세기 1:1의 대표 구절을 드립니다.
  태초에 하나님이 천지를 창조하시니라.

- 시작 부분의 요약 제공: 창세기 1장은 하나님이 six days에 걸쳐 세상을 창조하시는 이야기를 다룹니다. 빛과 어둠, 하늘과 바다, 땅과 식물, 해와 달과 별, 바다의 생물과 하늘의 새, 땅의 동물과 인간을 차례로 만들고, 마지막에 안식하시는 내용으로 시작합니다. 이어 창세기 2장에는 에덴 동산과 인간의 창조 및 선악과 이야기로 이어집니다.

원하시는 방식(발췌 또는 요약)이나 다른 번역(예: 영어 KJV)으로 들려드릴지 알려주시면 바로 제공해 드릴게요.

### UX를 개선한 챗봇 

- Streaming + Tools 챗봇 (web_search, mcp)
- 대화 히스토리를 list로 관리

In [4]:
tools = [
    {"type": "web_search"},
    {
        "type": "mcp",
        "server_label": "deepwiki",
        "server_url": "https://mcp.deepwiki.com/mcp",
        "require_approval": "never",
    },
]

conversation_history = []

print("=== Streaming Tools 챗봇 (web_search, mcp) ===")
print("사용 가능한 도구: 웹검색, DeepWiki(GitHub 저장소)")
print("종료하려면 'quit' 또는 'exit'를 입력하세요.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["quit", "exit"]:
        print("챗봇을 종료합니다.")
        break

    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    stream = client.responses.create(
        model=Model,
        input=conversation_history,
        tools=tools,
        stream=True,
    )

    print("Assistant: ", end="", flush=True)
    assistant_message = ""
    for chunk in stream:
        if chunk.type == "response.output_text.delta":
            print(chunk.delta, end="", flush=True)
            assistant_message += chunk.delta
    print("\n")

    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

=== Streaming Tools 챗봇 (web_search, mcp) ===
사용 가능한 도구: 웹검색, DeepWiki(GitHub 저장소)
종료하려면 'quit' 또는 'exit'를 입력하세요.



You:  ironmanciti/Generative_AI_3Days 에 있는 코드들의 목록과 README.md를 보여줘


Assistant: 다음은 ironmanciti/Generative_AI_3Days 저장소의 루트 디렉터리 파일/폴더 목록과 README.md 내용입니다.

루트에 있는 폴더 및 파일 목록
- .ipynb_checkpoints/ (폴더)
- data/ (폴더)
- db/ (폴더)
- images/ (폴더)
- output/ (폴더)
- 재무제표/ (폴더)
- .env_sample (파일)
- .gitignore (파일)
- 010_embedding.ipynb
- 150_autoregressive_language_generation.ipynb
- 151_autoregressive_language_generation_ClovaX.ipynb
- 200_Text_Generation-Prompts.ipynb
- 210_Structured_Output.ipynb
- 220_Function_Call.ipynb
- 225_Tools.ipynb
- 230_Conversation_state.ipynb
- 235_Reasoning.ipynb
- 250_Streaming.ipynb
- 310_Image_Generation.ipynb
- 320_Vision.ipynb
- 401_Embeddings_NaverMovie_Sentiment_Analysis.ipynb
- 700_Agents_Overview.ipynb
- 700_Agents_Overview.py
- Practice_answer_700_Agents_Overview.ipynb
- README.md
- README_agents.md
- Template_Creation.ipynb
- appscript-docs.js
- appscript-sheets.js
- appscript-slides.js
- cat_and_otter_realistic.png
- glide_out.png
- otter.png
- token_analysis.py
출처: GitHub 저장소의 루트 파일/폴더 목록 페이지. ([github.com](https://git

You:  exit


챗봇을 종료합니다.
